# 🌩️ Iowa City Weather Prediction Model
## Thesis Research Notebook — Scikit-Learn Predictive Modeling
### Location: Iowa City, IA 52246  |  41.6611°N, 91.5302°W

---

**Research Goals:**
1. Model **average daily weather patterns** (temperature, precipitation, wind)
2. Predict **storm frequency** and classify storm types
3. Compare multiple ML algorithms and evaluate them rigorously

**Data Source:** [Open-Meteo Historical Weather API](https://open-meteo.com) — Free, no API key, powered by ERA5 reanalysis from ECMWF

**Key ML Concepts Covered:**
- Feature Engineering (lag features, rolling stats, cyclical encoding)
- Regression vs Classification
- TimeSeriesSplit Cross-Validation (vs regular k-fold)
- Bias–Variance Tradeoff
- Class Imbalance handling
- Evaluation metrics: RMSE, MAE, R², F1, Confusion Matrix

---

## Section 1 — Import Required Libraries

We'll use:
- **`pandas` / `numpy`** — data manipulation and numerical computing
- **`matplotlib` / `seaborn`** — plotting (thesis-quality charts)
- **`requests`** — HTTP calls to the Open-Meteo API
- **`scikit-learn`** — the entire ML pipeline

> 📘 **Why scikit-learn?**  
> scikit-learn is the industry standard Python ML library. It provides a consistent API (`fit` → `predict` → `score`) across all algorithms, making it easy to swap models and compare them fairly.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from datetime import datetime, timedelta

# scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, roc_auc_score
)

# Plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_palette("tab10")

# ── Iowa City constants ────────────────────────────────────────────────────────
LAT       = 41.6611
LON       = -91.5302
CITY_NAME = "Iowa City, IA 52246"

print("✅ Libraries loaded successfully!")
print(f"   numpy     {np.__version__}")
print(f"   pandas    {pd.__version__}")

## Section 2 — Fetch Historical Weather Data for Iowa City, IA

We'll pull **15 years of daily weather data** (2010–present) from the **Open-Meteo Historical Archive API**.

### What is ERA5 Reanalysis?

> 📘 **ERA5** is produced by the European Centre for Medium-Range Weather Forecasts (ECMWF). Rather than raw station data, ERA5 runs a global atmospheric physics model and *assimilates* billions of observations — satellite retrievals, weather balloons (radiosondes), surface stations, aircraft reports — to produce a spatially complete, physically consistent reconstruction of the atmosphere at **~31 km resolution** every hour, going back to 1940.  
>
> This means we always get a value for Iowa City even on days with sparse local station coverage — crucial for long training datasets.

### Variables We're Fetching

| Variable | Description | Why It Matters |
|---|---|---|
| `temperature_2m_max/min/mean` | Air temp 2 m above ground | Core regression target |
| `apparent_temperature_*` | "Feels like" (heat index / wind chill) | Perceived comfort, energy demand |
| `precipitation_sum` | Total liquid + snow equivalent | Flood risk, agriculture |
| `rain_sum / snowfall_sum` | Phase-split precipitation | Snow vs rain classification |
| `windspeed_10m_max` | Peak 10-m wind speed | Storm severity proxy |
| `windgusts_10m_max` | Highest 3-second gust | Severe weather indicator |
| `weather_code` | WMO WW code (storm type) | Classification target |
| `shortwave_radiation_sum` | Incoming solar energy (MJ/m²) | Drives convection initiation |

In [ ]:
def fetch_weather_data(start_date="2010-01-01", end_date=None):
    """Download daily weather data from Open-Meteo Archive API."""
    if end_date is None:
        end_date = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":  LAT,
        "longitude": LON,
        "start_date": start_date,
        "end_date":   end_date,
        "daily": [
            "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
            "apparent_temperature_max", "apparent_temperature_min",
            "precipitation_sum", "rain_sum", "snowfall_sum",
            "precipitation_hours", "windspeed_10m_max", "windgusts_10m_max",
            "winddirection_10m_dominant", "shortwave_radiation_sum",
            "et0_fao_evapotranspiration", "weather_code",
        ],
        "timezone":           "America/Chicago",
        "temperature_unit":   "fahrenheit",
        "windspeed_unit":     "mph",
        "precipitation_unit": "inch",
    }

    print(f"📡 Fetching data for {CITY_NAME}  ({start_date} → {end_date}) ...")
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()

    df = pd.DataFrame(resp.json()["daily"])
    df["time"] = pd.to_datetime(df["time"])
    df.set_index("time", inplace=True)
    df.index.name = "date"

    print(f"✅ Retrieved {len(df):,} days  ({df.index[0].date()} → {df.index[-1].date()})")
    print(f"   Shape: {df.shape}")
    return df


df_raw = fetch_weather_data(start_date="2010-01-01")
df_raw.head()

## Section 3 — Exploratory Data Analysis (EDA)

Before building any model, we **always** explore the data first. EDA answers:
- What's the shape and structure of the dataset?
- Are there missing values?
- What's the distribution of each variable?
- Are there outliers?
- What are the seasonal patterns?

> 📘 **Why EDA matters for modeling:**  
> Garbage in = garbage out. A model trained on poorly understood data will produce unreliable predictions. EDA informs feature engineering decisions and helps catch data quality issues before they corrupt your results.

### WMO Weather Code Legend
The `weather_code` column uses standardized WMO codes:

| Code Range | Weather Type |
|---|---|
| 0–3 | Clear / Partly Cloudy |
| 45, 48 | Fog |
| 51–57 | Drizzle (light → freezing) |
| 61–67 | Rain (slight → freezing) |
| 71–77 | Snow / Ice Pellets |
| 80–82 | Rain Showers |
| 85–86 | Snow Showers |
| 95, 96, 99 | Thunderstorm (with/without hail) |

In [ ]:
# ── 3a. Basic dataset info ─────────────────────────────────────────────────────
print("=" * 55)
print("  DATASET INFO")
print("=" * 55)
df_raw.info()

print("\n" + "=" * 55)
print("  MISSING VALUES")
print("=" * 55)
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.any() else "  ✅ No missing values!")

print("\n" + "=" * 55)
print("  DESCRIPTIVE STATISTICS")
print("=" * 55)
df_raw.describe().round(2)

In [ ]:
# ── 3b. Storm type distribution (WMO codes) ───────────────────────────────────
WMO_MAP = {
    "Clear / Cloudy": list(range(0, 4)),
    "Fog":            [45, 48],
    "Drizzle":        list(range(51, 58)),
    "Rain":           list(range(61, 66)),
    "Freezing Rain":  [66, 67],
    "Snow":           list(range(71, 78)),
    "Rain Showers":   list(range(80, 83)),
    "Snow Showers":   [85, 86],
    "Thunderstorm":   [95, 96, 99],
}

def classify_weather(code):
    for label, codes in WMO_MAP.items():
        if int(code) in codes:
            return label
    return "Other"

df_raw["storm_category"] = df_raw["weather_code"].apply(classify_weather)

print("Storm category counts:")
counts = df_raw["storm_category"].value_counts()
total  = len(df_raw)
for cat, cnt in counts.items():
    bar = "█" * max(1, int(cnt / total * 40))
    print(f"  {cat:<18}  {cnt:5,d}  ({cnt/total*100:5.1f}%)  {bar}")

In [ ]:
# ── 3c. EDA Visualizations ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"EDA — {CITY_NAME}", fontsize=15, fontweight="bold")

df = df_raw.copy()
df["month"] = df.index.month
df["year"]  = df.index.year

# 1. Temperature time series + rolling mean
ax = axes[0, 0]
ax.fill_between(df.index, df["temperature_2m_min"], df["temperature_2m_max"],
                alpha=0.2, color="steelblue", label="Min–Max range")
ax.plot(df.index, df["temperature_2m_mean"].rolling(30, center=True).mean(),
        color="crimson", linewidth=1.5, label="30-day mean")
ax.set_title("Daily Temperature Range")
ax.set_ylabel("°F")
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# 2. Monthly temperature boxplot
ax = axes[0, 1]
monthly = [df[df["month"] == m]["temperature_2m_mean"].dropna().values for m in range(1, 13)]
bp = ax.boxplot(monthly, patch_artist=True)
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 12))
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.8)
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_title("Monthly Temperature Distribution")
ax.set_ylabel("Mean Temp (°F)")
ax.axhline(32, color="blue", linestyle="--", linewidth=0.8, alpha=0.5)

# 3. Annual precipitation
ax = axes[0, 2]
annual_precip = df.groupby("year")["precipitation_sum"].sum()
ax.bar(annual_precip.index, annual_precip.values, color="steelblue", alpha=0.7)
ax.axhline(annual_precip.mean(), color="red", linestyle="--", linewidth=1.5,
           label=f"Mean = {annual_precip.mean():.1f} in")
ax.set_title("Annual Precipitation Total")
ax.set_ylabel("Inches")
ax.legend()

# 4. Storm frequency bar
ax = axes[1, 0]
storm_counts = df["storm_category"].value_counts()
colors_bar   = plt.cm.tab10(np.linspace(0, 1, len(storm_counts)))
bars = ax.barh(storm_counts.index, storm_counts.values, color=colors_bar)
ax.set_title("Storm Category Frequency")
ax.set_xlabel("Days")
ax.invert_yaxis()

# 5. Precipitation distribution (log scale)
ax = axes[1, 1]
precip_nonzero = df[df["precipitation_sum"] > 0]["precipitation_sum"]
ax.hist(precip_nonzero, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
ax.set_title("Precipitation Distribution\n(rainy days only)")
ax.set_xlabel("Precipitation (in)")
ax.set_ylabel("Frequency")

# 6. Correlation heatmap
ax = axes[1, 2]
numeric_cols = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
                "precipitation_sum", "windspeed_10m_max", "shortwave_radiation_sum"]
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1,
            xticklabels=[c.replace("temperature_2m_", "temp_")
                          .replace("_sum","").replace("windspeed_10m_","wind_")
                          .replace("shortwave_radiation_","solar_") for c in numeric_cols],
            yticklabels=[c.replace("temperature_2m_", "temp_")
                          .replace("_sum","").replace("windspeed_10m_","wind_")
                          .replace("shortwave_radiation_","solar_") for c in numeric_cols])
ax.set_title("Feature Correlation Matrix")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
print("💾 Saved: eda_overview.png")
plt.show()

## Section 4 — Data Preprocessing & Feature Engineering

Raw sensor data is not model-ready. We need to:
1. **Handle missing values** — impute or drop
2. **Scale features** — many algorithms are distance-based and sensitive to magnitude
3. **Engineer new features** — create variables that help the model find patterns

### ① Lag Features
> 📘 **Autocorrelation:** In time-series, today's weather is strongly correlated with yesterday's — this is autocorrelation. A lag-1 feature captures "what was yesterday's temperature?". Lag-7 captures last week. These give the model memory.

### ② Rolling Statistics
> 📘 **Rolling mean/sum:** A 7-day rolling average smooths noise and captures recent trends. A 30-day rolling precipitation sum approximates soil moisture — important for convective storm initiation (wet ground → more evaporation → more instability).

### ③ Cyclical Encoding of Day-of-Year
> 📘 **The wrapping problem:** If we use day_of_year = 1…365 directly, the model thinks Dec 31 (day 365) and Jan 1 (day 1) are 364 days apart — but physically they're adjacent! We fix this with sine/cosine encoding:  
> $$\text{sin\_doy} = \sin\!\left(\frac{2\pi \cdot \text{doy}}{365.25}\right), \quad \text{cos\_doy} = \cos\!\left(\frac{2\pi \cdot \text{doy}}{365.25}\right)$$  
> This maps the year to a circle where Jan 1 and Dec 31 are neighbors. ✓

### ④ Interaction Terms
> 📘 **Wind × Precipitation:** A linear model can't naturally learn "heavy rain at high wind speed = more dangerous" unless we explicitly create that product as a feature. Interaction terms let linear models approximate non-linear relationships.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build feature matrix from raw daily weather data."""
    df = df.copy()

    # ── Derived features ──────────────────────────────────────────────────────
    df["temp_range"]       = df["temperature_2m_max"] - df["temperature_2m_min"]
    df["feels_like_range"] = df["apparent_temperature_max"] - df["apparent_temperature_min"]

    # ── ① Lag features ────────────────────────────────────────────────────────
    for lag in [1, 2, 3, 7, 14]:
        df[f"temp_mean_lag{lag}"] = df["temperature_2m_mean"].shift(lag)
        df[f"precip_lag{lag}"]    = df["precipitation_sum"].shift(lag)
        df[f"wind_lag{lag}"]      = df["windspeed_10m_max"].shift(lag)

    # ── ② Rolling statistics ──────────────────────────────────────────────────
    for w in [3, 7, 14, 30]:
        df[f"temp_roll{w}"]   = df["temperature_2m_mean"].rolling(w).mean()
        df[f"precip_roll{w}"] = df["precipitation_sum"].rolling(w).sum()
        df[f"wind_roll{w}"]   = df["windspeed_10m_max"].rolling(w).mean()

    # ── ③ Cyclical time encoding ──────────────────────────────────────────────
    doy = df.index.dayofyear
    df["sin_doy"] = np.sin(2 * np.pi * doy / 365.25)
    df["cos_doy"] = np.cos(2 * np.pi * doy / 365.25)
    df["month"]   = df.index.month
    df["year"]    = df.index.year

    # ── Iowa seasons (one-hot) ────────────────────────────────────────────────
    season_map = {12:"Winter",1:"Winter",2:"Winter",
                  3:"Spring",4:"Spring",5:"Spring",
                  6:"Summer",7:"Summer",8:"Summer",
                  9:"Fall",10:"Fall",11:"Fall"}
    df["season"] = df.index.month.map(season_map)
    dummies = pd.get_dummies(df["season"], prefix="season")
    df = pd.concat([df, dummies], axis=1)

    # ── ④ Interaction terms ───────────────────────────────────────────────────
    df["wind_x_precip"] = df["windspeed_10m_max"] * df["precipitation_sum"]
    df["temp_x_precip"] = df["temperature_2m_mean"] * df["precipitation_sum"]
    df["precip_flag"]   = (df["precipitation_sum"] > 0.01).astype(int)

    # Storm label
    df["storm_category"] = df["weather_code"].apply(classify_weather)

    return df


df_feat = engineer_features(df_raw)

print(f"Feature matrix: {df_raw.shape[1]} raw columns → {df_feat.shape[1]} total columns")
print(f"Rows with any NaN (from lag/rolling): {df_feat.isnull().any(axis=1).sum():,}")
print(f"\nNew engineered features (sample):")
new_cols = [c for c in df_feat.columns if c not in df_raw.columns]
for c in new_cols[:15]:
    print(f"  {c}")

## Section 5 — Visualizing Weather Patterns & Storm Frequencies

These charts are designed to be **thesis-ready** — they reveal patterns in Iowa City's climate that contextualise our models.

> 📘 **Climatology vs Forecast:**  
> This section shows *climatology* — the long-run statistical distribution of weather. Before building a forecast model, you need to understand climatology because:
> - It defines your **baseline** (a climatological mean is the simplest possible "model")
> - It reveals **seasonality** your features must capture  
> - It quantifies **class imbalance** in storm events

In [ ]:
# ── 5a. Storm seasonality heatmap ─────────────────────────────────────────────
storm_cats = ["Thunderstorm","Rain","Freezing Rain","Snow","Drizzle","Rain Showers","Snow Showers"]
df_feat["month"] = df_feat.index.month
df_feat["year"]  = df_feat.index.year

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Storm Seasonality — {CITY_NAME}", fontsize=14, fontweight="bold")

# Heatmap: storm type × month
pivot = (df_feat[df_feat["storm_category"].isin(storm_cats)]
         .pivot_table(values="weather_code", index="storm_category",
                      columns="month", aggfunc="count", fill_value=0))
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
sns.heatmap(pivot, ax=axes[0], cmap="YlOrRd", annot=True, fmt="d",
            linewidths=0.5, cbar_kws={"label": "Days/year (avg)"})
axes[0].set_title("Storm Events by Month")
axes[0].set_ylabel("")

# Annual stacked bar
annual = (df_feat[df_feat["storm_category"].isin(storm_cats)]
          .groupby(["year","storm_category"]).size().unstack(fill_value=0))
annual.plot(kind="bar", ax=axes[1], stacked=True, colormap="tab10")
axes[1].set_title("Annual Storm Event Counts")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Days")
axes[1].legend(fontsize=8, bbox_to_anchor=(1.01,1), loc="upper left")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.savefig("storm_seasonality.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5b. Precipitation calendar heatmap ────────────────────────────────────────
pivot_precip = df_feat.pivot_table(
    values="precipitation_sum", index="year", columns="month", aggfunc="sum"
)
pivot_precip.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                        "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_precip, ax=ax, cmap="Blues", annot=True, fmt=".1f",
            linewidths=0.4, cbar_kws={"label": "Total Precip (in)"})
ax.set_title(f"Monthly Precipitation Totals — {CITY_NAME}", fontsize=13, fontweight="bold")
ax.set_ylabel("Year")
plt.tight_layout()
plt.savefig("precip_calendar.png", dpi=150, bbox_inches="tight")
plt.show()

# Key patterns to note:
print("📊 Precipitation insights:")
print(f"  Wettest month on average: {pivot_precip.mean().idxmax()}")
print(f"  Driest month on average:  {pivot_precip.mean().idxmin()}")
print(f"  Total avg annual precip:  {pivot_precip.sum(axis=1).mean():.1f} in")

## Section 6 — Encoding & Temporal Train/Test Split

### Why Temporal Splitting Matters

> 📘 **The golden rule of time-series ML: never let the model see the future.**  
> Using `train_test_split(shuffle=True)` on time-series data would mix January 2025 data into the training set while testing on December 2024 — called **data leakage**. The model appears to perform well but would fail completely in deployment.  
>
> Correct approach: **always split by time**. Train on earlier years, test on later years.

```
Timeline: ──────────────────────────────────────────────────────
           2010                          2021          2024
           |──────── TRAINING (80%) ──────|── TEST (20%) ──|
```

### Label Encoding for Storm Classes

> 📘 **LabelEncoder** maps string categories → integers: `{"Clear":0, "Rain":1, ...}`. Required because scikit-learn models work only with numbers. For classification with tree models, label encoding is fine. For distance-based models (SVC, KNN), prefer one-hot encoding to avoid implying ordinal relationships between classes.

In [ ]:
# ── Prepare clean dataset ─────────────────────────────────────────────────────
df_model = df_feat.dropna().copy()

# Columns to exclude (raw targets or non-numeric labels)
EXCLUDE = {
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "apparent_temperature_max", "apparent_temperature_min",
    "precipitation_sum", "rain_sum", "snowfall_sum",
    "weather_code", "storm_category", "season",
}
numeric_dtypes = [np.float64, np.int64, bool, np.uint8]
FEATURE_COLS = [c for c in df_model.columns
                if c not in EXCLUDE and df_model[c].dtype in numeric_dtypes]

print(f"✅ Feature columns ({len(FEATURE_COLS)} total):")
for c in FEATURE_COLS:
    print(f"   {c}")

# ── Temporal train/test split (80/20) ─────────────────────────────────────────
SPLIT_N = int(len(df_model) * 0.80)

X_all = df_model[FEATURE_COLS]
X_train_raw = X_all.iloc[:SPLIT_N]
X_test_raw  = X_all.iloc[SPLIT_N:]

print(f"\n📅 Train: {X_train_raw.index[0].date()} → {X_train_raw.index[-1].date()} ({len(X_train_raw):,} days)")
print(f"📅 Test:  {X_test_raw.index[0].date()}  → {X_test_raw.index[-1].date()} ({len(X_test_raw):,} days)")

# ── StandardScaler (fit ONLY on training data) ────────────────────────────────
# [Important: never fit scaler on test data — that would be leakage]
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_test_sc  = scaler.transform(X_test_raw)

# ── Storm classification label encoding ───────────────────────────────────────
df_clf = df_model[df_model["storm_category"] != "Other"].copy()
le     = LabelEncoder()
y_clf  = le.fit_transform(df_clf["storm_category"])
X_clf  = df_clf[FEATURE_COLS]
n_train_clf = int(len(X_clf) * 0.80)
X_clf_train  = X_clf.iloc[:n_train_clf]
X_clf_test   = X_clf.iloc[n_train_clf:]
y_clf_train  = y_clf[:n_train_clf]
y_clf_test   = y_clf[n_train_clf:]

print(f"\nStorm classes: {list(le.classes_)}")

## Section 7 — Train Baseline Predictive Models

We train **four regression models** to predict next-day maximum temperature, then **three classifiers** for storm type.

### The Models — Mathematical Intuition

#### 📘 Ridge Regression (L2-Regularised Linear Model)
Minimises: $\mathcal{L} = \sum_i (y_i - \hat{y}_i)^2 + \alpha \|\mathbf{w}\|_2^2$

The $\alpha \|\mathbf{w}\|_2^2$ penalty shrinks coefficients toward zero, preventing overfitting on noisy features. Best interpretability of all models.

#### 📘 Random Forest
- Trains $T$ decision trees, each on a **bootstrap sample** (random 63% of rows) with a **random subset of features** at each split
- Prediction = average of all tree outputs (regression) or majority vote (classification)  
- The randomness decorrelates the trees, making the ensemble much more robust than any single tree

#### 📘 Gradient Boosting
Builds trees **sequentially**. Each tree $t$ fits the **residual errors** of the previous ensemble:
$$F_t(x) = F_{t-1}(x) + \eta \cdot h_t(x)$$
where $\eta$ is the learning rate (shrinkage). Often the highest accuracy but can overfit without proper regularization.

#### 📘 SVR / SVC (Support Vector Machine)
Finds a hyperplane that:
- **SVR:** keeps predictions within an $\varepsilon$-tube around true values
- **SVC:** maximises margin between classes

The **kernel trick** ($K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$) implicitly maps data to higher-dimensional space without computing the transform explicitly.

In [ ]:
# ── 7a. Regression: Predict next-day Max Temperature ─────────────────────────
TARGET    = "temperature_2m_max"
HORIZON   = 1   # days ahead

y_all     = df_model[TARGET].shift(-HORIZON)
valid     = ~y_all.isna()
y_all     = y_all[valid]
X_v       = df_model[FEATURE_COLS][valid]
split_n   = int(len(X_v) * 0.80)
y_train   = y_all.iloc[:split_n]
y_test    = y_all.iloc[split_n:]
X_tr_raw  = X_v.iloc[:split_n]
X_te_raw  = X_v.iloc[split_n:]

sc2 = StandardScaler()
X_tr_sc = sc2.fit_transform(X_tr_raw)
X_te_sc = sc2.transform(X_te_raw)

REGRESSORS = {
    "Ridge":             Ridge(alpha=1.0),
    "Random Forest":     RandomForestRegressor(n_estimators=300, max_depth=12,
                                               min_samples_leaf=3, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                   max_depth=5, subsample=0.8, random_state=42),
    "SVR":               SVR(kernel="rbf", C=10, epsilon=0.5),
}
NEEDS_SCALE = {"Ridge", "SVR"}
tscv = TimeSeriesSplit(n_splits=5)

reg_results = {}
print(f"{'Model':<22}  {'RMSE':>6}  {'MAE':>6}  {'R²':>7}  {'CV-R²':>12}")
print("─" * 62)

for name, model in REGRESSORS.items():
    tr = X_tr_sc if name in NEEDS_SCALE else X_tr_raw.values
    te = X_te_sc if name in NEEDS_SCALE else X_te_raw.values
    cv = cross_val_score(model, tr, y_train, cv=tscv, scoring="r2", n_jobs=-1)
    model.fit(tr, y_train)
    y_pred = model.predict(te)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    reg_results[name] = {"model": model, "y_pred": y_pred,
                         "rmse": rmse, "mae": mae, "r2": r2,
                         "cv_r2": cv.mean(), "cv_r2_std": cv.std()}
    print(f"{name:<22}  {rmse:6.3f}  {mae:6.3f}  {r2:7.4f}  {cv.mean():6.4f}±{cv.std():.4f}")

best_reg = min(reg_results, key=lambda k: reg_results[k]["rmse"])
print(f"\n🏆 Best regressor: {best_reg}  (RMSE = {reg_results[best_reg]['rmse']:.3f} °F)")

In [ ]:
# ── 7b. Storm Classification ──────────────────────────────────────────────────
# [📘 Class Imbalance]
#   Thunderstorms are rare — ~7% of days. A "dumb" model that predicts
#   "Clear" every day would get 55% accuracy but detect ZERO storms.
#   class_weight='balanced' gives rare classes proportionally higher loss weight.

sc_clf = StandardScaler()
X_clf_tr_sc = sc_clf.fit_transform(X_clf_train)
X_clf_te_sc = sc_clf.transform(X_clf_test)

CLASSIFIERS = {
    "Random Forest":     RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                                    max_depth=5, random_state=42),
    "SVC":               SVC(kernel="rbf", C=10, class_weight="balanced", probability=True),
}

clf_results = {}
print(f"{'Model':<22}  {'Weighted F1':>12}  {'CV F1':>14}")
print("─" * 55)

for name, model in CLASSIFIERS.items():
    tr = X_clf_tr_sc if name == "SVC" else X_clf_train.values
    te = X_clf_te_sc if name == "SVC" else X_clf_test.values
    cv = cross_val_score(model, tr, y_clf_train, cv=tscv, scoring="f1_weighted", n_jobs=-1)
    model.fit(tr, y_clf_train)
    y_pred = model.predict(te)
    f1 = f1_score(y_clf_test, y_pred, average="weighted")
    clf_results[name] = {"model": model, "y_pred": y_pred,
                         "f1": f1, "cv_f1": cv.mean(), "cv_f1_std": cv.std()}
    print(f"{name:<22}  {f1:12.4f}  {cv.mean():6.4f}±{cv.std():.4f}")

best_clf = max(clf_results, key=lambda k: clf_results[k]["f1"])
print(f"\n🏆 Best classifier: {best_clf}  (F1 = {clf_results[best_clf]['f1']:.4f})")

print(f"\n{'─'*55}")
print(f"Classification Report — {best_clf}")
print(f"{'─'*55}")
print(classification_report(y_clf_test, clf_results[best_clf]["y_pred"],
                             target_names=le.classes_, zero_division=0))

## Section 8 — Hyperparameter Tuning with GridSearchCV

> 📘 **Hyperparameters** are settings you choose *before* training — the model doesn't learn them from data (unlike weights). Examples:
> - `n_estimators` — how many trees in a forest
> - `max_depth` — how deep each tree can grow
> - `learning_rate` — step size in gradient boosting

### The Bias–Variance Tradeoff

$$\text{Expected Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

| | Too Simple (High Bias) | Too Complex (High Variance) |
|---|---|---|
| Training error | High | Very Low |
| Test error | High | High (overfitting) |
| Example | `max_depth=1` | `max_depth=100` |

> 📘 **GridSearchCV** exhaustively tests all combinations in a parameter grid and picks the one with the best cross-validated score. We use `TimeSeriesSplit` as the CV strategy to avoid temporal leakage.

In [ ]:
# ── GridSearchCV: Tune Random Forest Regressor ────────────────────────────────
# Using a reduced grid here for speed — expand for thesis final run
param_grid = {
    "n_estimators": [100, 200],
    "max_depth":    [8, 12, 16],
    "min_samples_leaf": [2, 5],
}

print("🔍 Running GridSearchCV on Random Forest Regressor...")
print("   (This searches 12 parameter combinations × 3 CV folds = 36 fits)")

tscv3 = TimeSeriesSplit(n_splits=3)   # fewer folds for speed
gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=tscv3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=0,
)
gs.fit(X_tr_raw.values, y_train)

print(f"\n✅ Best parameters: {gs.best_params_}")
print(f"   Best CV RMSE: {-gs.best_score_:.3f} °F")

best_tuned_rf = gs.best_estimator_
y_pred_tuned  = best_tuned_rf.predict(X_te_raw.values)
rmse_tuned    = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned      = r2_score(y_test, y_pred_tuned)
print(f"   Test RMSE after tuning: {rmse_tuned:.3f} °F")
print(f"   Test R² after tuning:   {r2_tuned:.4f}")
print(f"\n   Baseline RF RMSE:  {reg_results['Random Forest']['rmse']:.3f} °F")
print(f"   Improvement:       {reg_results['Random Forest']['rmse'] - rmse_tuned:+.3f} °F")

## Section 9 — Evaluate Model Performance

### Regression Metrics — What Do They Mean?

| Metric | Formula | Interpretation |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(\hat{y}-y)^2}$ | In °F — penalises large errors heavily. Good for safety-critical forecasting. |
| **MAE** | $\frac{1}{n}\sum|\hat{y}-y|$ | In °F — average absolute error, easy to interpret. |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Fraction of variance explained. R²=0.90 → model explains 90% of temperature variability. |

### Classification Metrics

> 📘 **Precision vs Recall tradeoff for storms:**  
> - High **Recall** = model catches most real storms (few missed events). Critical for public safety!  
> - High **Precision** = when model says "storm", it's usually right (few false alarms)  
> - **F1** = harmonic mean of both; good single metric for imbalanced classes  
>
> For emergency management, you'd **prioritise recall** — a missed tornado warning is far more costly than a false alarm.

In [ ]:
# ── 9a. Regression: predicted vs actual scatter grid ─────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle(f"Next-Day Max Temp Prediction — Model Comparison\n{CITY_NAME}",
             fontsize=13, fontweight="bold")

color_map = {"Ridge":"#4E79A7","Random Forest":"#F28E2B",
             "Gradient Boosting":"#E15759","SVR":"#76B7B2"}

for idx, (name, rm) in enumerate(reg_results.items()):
    ax = axes[idx // 2][idx % 2]
    y_pred_i = rm["y_pred"]
    lo = min(y_test.min(), y_pred_i.min()) - 3
    hi = max(y_test.max(), y_pred_i.max()) + 3
    ax.scatter(y_test, y_pred_i, alpha=0.3, s=7,
               color=color_map.get(name,"gray"), rasterized=True)
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.2, label="Perfect")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("Actual (°F)"); ax.set_ylabel("Predicted (°F)")
    ax.set_title(f"{name}\nRMSE={rm['rmse']:.2f}°F  R²={rm['r2']:.3f}")
    ax.legend(fontsize=8); ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.savefig("regression_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 9b. Feature importance ────────────────────────────────────────────────────
# [📘 Mean Decrease in Impurity (MDI)]
#   At each tree split, impurity (variance for regression) decreases.
#   MDI = total decrease credited to a feature across all trees.
#   Normalised to sum = 1.  High → model relies heavily on that feature.

rf_model = reg_results["Random Forest"]["model"]
fi = pd.DataFrame({"feature": FEATURE_COLS,
                   "importance": rf_model.feature_importances_}) \
       .sort_values("importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi["feature"][::-1], fi["importance"][::-1],
        color=plt.cm.viridis(np.linspace(0.2, 0.8, len(fi))))
ax.set_title("Top-20 Feature Importances (Random Forest)\nNext-Day Max Temp Prediction",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Importance (Mean Decrease in Impurity)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 10 features:")
for _, row in fi.head(10).iterrows():
    print(f"  {row['feature']:<30}  {row['importance']:.4f}")

In [ ]:
# ── 9c. Confusion matrix for storm classifier ─────────────────────────────────
# [📘 Confusion Matrix]
#   Rows = Actual class.  Columns = Predicted class.
#   Diagonal = correct.  Off-diagonal = errors.
#   For storms: a missed "Thunderstorm" (false negative) = safety risk!

best_clf_preds = clf_results[best_clf]["y_pred"]
cm = confusion_matrix(y_clf_test, best_clf_preds)

fig, ax = plt.subplots(figsize=(11, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=45)
ax.set_title(f"Storm Classification — Confusion Matrix\n({best_clf})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-class analysis
print("Per-class F1 scores:")
from sklearn.metrics import f1_score as f1_per
for i, cls in enumerate(le.classes_):
    mask = y_clf_test == i
    if mask.sum() == 0:
        continue
    class_f1 = f1_score(y_clf_test[mask], best_clf_preds[mask],
                        labels=[i], average="micro", zero_division=0)
    n = mask.sum()
    print(f"  {cls:<18}  n={n:4d}  F1={class_f1:.3f}")

## Section 10 — Predicting Future Weather Patterns

Now we use the best-trained models to:
1. Forecast **next-day max temperature** over the test period and compare to actual
2. Visualise **storm classification predictions** on a monthly grid

> 📘 **Model Limitations to Address in Your Thesis:**
> - Models are trained on historical ERA5 data — they may not generalise to rare "black swan" weather events outside the training distribution
> - We predict 1 day ahead; error grows exponentially at longer horizons (this is the *butterfly effect* in chaotic systems)
> - Feature lag structure means the model cannot be used beyond the training horizon without a rolling forecast strategy
> - Climate change may shift statistical distributions over time — a model trained on 2010–2020 data may degrade on 2030+ data

> 📘 **Potential Thesis Extensions:**
> - Add ENSO (El Niño/La Niña) indices as features — they modulate Iowa's seasonal weather
> - Try LSTM or Transformer neural networks for sequence modeling
> - Compare Open-Meteo ERA5 with NOAA ASOS station data from the Iowa City Municipal Airport
> - Build a 7-day rolling forecast by feeding predictions back as lag features

In [ ]:
# ── 10a. Predicted vs actual temperature time-series ─────────────────────────
best_rf_preds = reg_results[best_reg]["y_pred"]
test_dates    = X_te_raw.index

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.suptitle(f"Next-Day Max Temperature: Predicted vs Actual\n{CITY_NAME} — Test Set",
             fontsize=14, fontweight="bold")

# Full time series
ax = axes[0]
ax.plot(test_dates, y_test.values, color="steelblue", linewidth=0.9, label="Actual", alpha=0.85)
ax.plot(test_dates, best_rf_preds, color="crimson", linewidth=0.9, label=f"Predicted ({best_reg})", alpha=0.85)
ax.set_ylabel("Temperature (°F)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Residuals
residuals = y_test.values - best_rf_preds
ax = axes[1]
ax.bar(test_dates, residuals, color=np.where(residuals > 0, "steelblue", "crimson"), alpha=0.6, width=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(np.std(residuals), color="gray", linestyle="--", linewidth=0.8, label=f"±1σ = {np.std(residuals):.2f}°F")
ax.axhline(-np.std(residuals), color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Residual (Actual − Predicted, °F)")
ax.set_xlabel("Date")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("predicted_vs_actual_temp.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Residual statistics:")
print(f"  Mean:  {residuals.mean():+.3f} °F  (bias)")
print(f"  Std:   {residuals.std():.3f} °F")
print(f"  Max overestimate:   {residuals.min():+.2f} °F")
print(f"  Max underestimate:  {residuals.max():+.2f} °F")

In [ ]:
# ── 10b. Final model summary table ───────────────────────────────────────────
print("=" * 65)
print(f"  FINAL MODEL SUMMARY — {CITY_NAME}")
print("=" * 65)

print(f"\n  REGRESSION (Target: Next-Day Max Temperature)")
print(f"  {'Model':<22}  {'RMSE':>6}  {'MAE':>6}  {'R²':>7}  {'CV-R²':>12}")
print(f"  {'─'*22}  {'─'*6}  {'─'*6}  {'─'*7}  {'─'*12}")
for name, rm in sorted(reg_results.items(), key=lambda x: x[1]["rmse"]):
    marker = "🏆" if name == best_reg else "  "
    print(f"  {marker}{name:<20}  {rm['rmse']:6.3f}  {rm['mae']:6.3f}  {rm['r2']:7.4f}  "
          f"{rm['cv_r2']:6.4f}±{rm['cv_r2_std']:.4f}")

print(f"\n  CLASSIFICATION (Target: Storm Type)")
print(f"  {'Model':<22}  {'F1 (Weighted)':>14}  {'CV-F1':>14}")
print(f"  {'─'*22}  {'─'*14}  {'─'*14}")
for name, cm_r in sorted(clf_results.items(), key=lambda x: -x[1]["f1"]):
    marker = "🏆" if name == best_clf else "  "
    print(f"  {marker}{name:<20}  {cm_r['f1']:14.4f}  "
          f"{cm_r['cv_f1']:6.4f}±{cm_r['cv_f1_std']:.4f}")

print(f"\n  Tuned RF RMSE: {rmse_tuned:.3f} °F  (after GridSearchCV)")
print(f"\n{'=' * 65}")
print("  Charts saved:")
for f in ["eda_overview.png","storm_seasonality.png","precip_calendar.png",
          "regression_comparison.png","feature_importance.png",
          "confusion_matrix.png","predicted_vs_actual_temp.png"]:
    print(f"    📊 {f}")
print("=" * 65)